<a href="https://colab.research.google.com/github/m-akintunde/INM434/blob/main/lab9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
__author__ = "Michael Akintunde"
__version__ = "INM434/IN3045 City, University of London, Spring 2026"

Mostly the same setup code as in a previous lab.

In [2]:
# setting it all up
!pip install transformers accelerate torch --quiet

In [3]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoModel, AutoTokenizer
import warnings
warnings.filterwarnings("ignore")

# Check GPU
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
      if torch.cuda.is_available() else "")


GPU: NVIDIA L4
Memory: 23.7 GB


In [4]:
# Uncomment the following lines if you are having issues with models not
# downloading from HF Hub in subsequent cells.
# It requires a user access token, which can be obtained
# from the Hugging Face website.

# from huggingface_hub import login
# login("<PASTE-HF-USER-ACCESS-TOKEN-HERE>")

In [5]:
# As above, uncomment the following line in case of issues with downloading HF
# models.

# !rm -rf ~/.cache/huggingface

We will also be loading an embedding model, `Qwen3-Embedding-0.6B`.

In [6]:
MODEL_INSTRUCT = "Qwen/Qwen2.5-3B-Instruct"
MODEL_EMBED = "Qwen/Qwen3-Embedding-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_INSTRUCT)
model_instruct = AutoModelForCausalLM.from_pretrained(
    MODEL_INSTRUCT, torch_dtype=torch.float16, device_map="auto"
)

tokenizer_embed = AutoTokenizer.from_pretrained(MODEL_EMBED)
model_embed = AutoModel.from_pretrained(
    MODEL_EMBED, torch_dtype=torch.float16, device_map="auto"
)



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [7]:
!pip install langchain pypdf langchain_text_splitters langchain_community

In [8]:
!pip install pymongo

In [9]:
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

Mount your Google Drive. Go into your Google drive, create a folder called "documents" with the two papers and the .pem file (see instructions) before running these cells.

In [10]:
from google.colab import drive
drive.mount('/mnt/drive')

Drive already mounted at /mnt/drive; to attempt to forcibly remount, call drive.mount("/mnt/drive", force_remount=True).


In [11]:
import os
PATH_PREFIX = '/mnt/drive/MyDrive/'

In [16]:
import json
import base64
import ssl
import uuid
import re
import tiktoken
import langchain_text_splitters
import langchain_community
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)  # for exponential backoff

class ChunkerEmbedder:

    def __init__(self):

        # Replace these lines with those obtain from Mongo DB when creating
        # DB users (see instructions document).
        self.uri = "mongodb+srv://<cluster-name>.5quux.mongodb.net/?authSource=%24external&authMechanism=MONGODB-X509&appName=<cluster-name>"
        self.client = MongoClient(self.uri,
                                  tls=True,
                                  tlsCertificateKeyFile=os.path.join(PATH_PREFIX,'<path-to-certificate>'),
                                  server_api=ServerApi('1'))

        self.db = self.client.get_database('NLP')

        # 1024 is the default embedding dimension of Qwen3-Embedding-0.6B
        self.embedding_dimension = 1024
        self.chunk_encoder = model_embed
        self.history = []
        self.questions = []
        self.all_model_details = {
            "Qwen/Qwen2.5-3B": {
                "type": "causal",
                "max_toks": 32768,
            }, "Qwen/Qwen2.5-3B-Instruct": {
                "type": "causal",
                "max_toks": 32768,
            },  "Qwen/Qwen3-Embedding-0.6B": {
                "type": "text-embedding",
                "max_toks": 32768,
            }
        }

    def chunking_with_split(self, text_list_chunks, count_split=0,
                            limit=5000000):
        tokens_used = [len(chunk) for chunk in text_list_chunks]

        if sum(tokens_used) * 4 >= limit:
            print(
                f" Documents: {len(text_list_chunks)} exceeding tokens limit, splitting {sum(tokens_used) * 4}")
            split = int(len(text_list_chunks) / 2)
            return self.chunking_with_split(text_list_chunks[:split], count_split + 1) + self.chunking_with_split(
                text_list_chunks[split:], count_split + 2)
        else:
            print(f'Embedding chunks of length {len(text_list_chunks)}')
            inputs = tokenizer(text_list_chunks, return_tensors="pt").to(model_embed.device)
            embed = self.chunk_encoder(**inputs)
            if embed is None:
                return None
            if count_split > 0:
                time.sleep(60)

            embed = embed.last_hidden_state[:, 0, :]

            return F.normalize(embed, p=2, dim=1)

    def chunk_document(self, doc, max_tokens=200):
        file_uid = uuid.uuid4()

        # This function will be called on both string and lists of strings.
        if isinstance(doc, list):
            doc = "\n".join([str(x) for x in doc])

        elif not isinstance(doc, str):
            doc = str(doc)

        text_splitter = langchain_text_splitters.RecursiveCharacterTextSplitter(
            separators=["\n\n", "\n", ".", " "],
            chunk_size=200,
            chunk_overlap=20)

        texts = text_splitter.split_text(doc)

        text_filtered = [
            text for text in texts
            if text.strip() != "" and len(text.strip()) > 10
        ]

        text_list_chunks = [text.replace("\n", " ") for text in text_filtered]
        embeds = self.create_embeddings(list(text_list_chunks))
        if embeds is not None:

            data = []
            for i, text in enumerate(text_list_chunks):
                data.append({
                    'content': text,
                    'chunk_id': str(uuid.uuid4()),
                    'doc_id': str(file_uid),
                    'embeddings': embeds[i]
                })

            self.db.documents.insert_many(data)
            print(f"Inserted {len(data)} chunks")
            return data

    @retry(wait=wait_random_exponential(min=1, max=10), stop=stop_after_attempt(6))
    def create_embeddings(self, text_list, batch_size=8):
        ''' Create embeddings for a list of text

        Args:
        - text_list: A list of text strings for which embeddings are to be created.

        Returns:
        - A list containing the embeddings for each text string in the input list.

        '''

        all_embeddings = []

        # Uses batches, in case encountering OOM errors, commonly seen when
        # using L4 runtimes.
        for i in range(0, len(text_list), batch_size):
            batch = text_list[i:i+batch_size]

            # Sometimes will be a dict when retriving results via vector search.
            batch = [x["content"] if isinstance(x, dict) else x for x in batch]
            batch = [text.replace("\n", " ") for text in batch]
            batch = [f"Instruct: Represent the sentence for retrieval\nQuery: {text}" for text in batch]
            try:
                assert all(isinstance(x, str) for x in batch), batch
                # Tokenize inputs
                inputs = tokenizer_embed(batch,
                                  return_tensors="pt",
                                  padding=True,
                                  truncation=True,
                                  max_length=512)

                # After tokenizing, move tensors to GPU.
                inputs = {k: v.to(model_embed.device) for k, v in inputs.items()}

                with torch.no_grad():
                    # Embed them
                    results = model_embed(**inputs)

                # Extract the embedding. The Qwen3 Embedding model uses last
                # token pooling.
                embed = results.last_hidden_state[:, -1]

                # TODO: What should we do here to the embedding?

                all_embeddings.append(embed.cpu())
            except torch.cuda.OutOfMemoryError:
                print("CUDA OOM - clear cache and try smaller batch size")
                torch.cuda.empty_cache()
                import gc
                gc.collect()
                if batch_size == 1:
                    raise
                return self.create_embeddings(text_list[i:i+batch_size], batch_size=batch_size // 2)
            finally:
                pass


        # Move tensor from GPU to CPU, and convert to numpy array.
        # Then convert to list, since MongoDB expects lists.
        return torch.cat(all_embeddings, dim=0).cpu().numpy().tolist()

    def vector_search(self, query, tot_docs=15):
        '''
        Args:
        - query (str): The search query for which relevant documents are to be
                       retrieved.
        - tot_docs (int, optional): The maximum number of documents to return
                                    initially. Defaults to 15.

        Returns:
        - position_score (list[dict]): The full response from the vector search
                                       including score.
        - position_list (list[int]): A list of positions indicating the
                                     locations of the relevant documents in the
                                     database.
        '''

        query_vector = self.create_embeddings([query])[0]

        # In case vector search rejects numpy types
        query_vector = list(map(float, query_vector))

        vector_search_stage = {
            '$vectorSearch': {
                'queryVector': query_vector,
                "path": "embeddings",
                'numCandidates': 500,
                "index": "vec_idx",
                'limit': tot_docs,
            }
        }

        pipeline = [
            vector_search_stage,
            {"$set": {"score": {"$meta": "vectorSearchScore"}}},
            {
                '$unset': 'embeddings'
            }, {
                '$project': {
                    '_id': 0,
                    'doc_id': "$doc_id",
                    'score': "$score",
                    'content': '$content',
                    'embeddings': '$embeddings'
                }
            }, {
                '$sort': {
                    'score': -1
                }
            }
        ]

        results = list(self.db.documents.aggregate(pipeline))
        doc_ids = [result['doc_id'] for result in results]
        contents = [result['content'] for result in results]
        return results, doc_ids, contents

    def create_prompt_text(self, full_context: list):
        tmp = ""
        concatenated_context = []
        for context in full_context:
            tmp += f"{context}"
            concatenated_context.append(tmp)
            tmp = ""

        separator = "\n\n" + "-" * 50 + "\n\n"
        prompt_context = (separator).join(concatenated_context)
        return prompt_context

    def trim_snippet(self, snippet: str):
        # Encode text to tokens
        tokens = tokenizer.encode(snippet)

        # Collect the last 20 tokens
        trimmed_tokens = tokens[:-20]

        # Decode them to get back to text
        return tokenizer.decode(trimmed_tokens)

    def ask_question(self, query, prompt_context_str, model_params,
                     filtered_question=True, ignore_history: bool = False,
                     prompt_template: str = None):
        model_details = self.all_model_details[model_params['model']]
        result, history = self.get_filter_answer_llm(query, prompt_context_str,
                                                     model_params['model'],
                                                     ignore_history)
        data = {
            "result": result,
            "history": history,
            "prompt_context": prompt_context_str,
            "model_details": model_details,
            'response_app': [{
                'question': query,
                'cot': result['chain of thought'] if 'chain of thought' in result else "",
                'answer': result['answer'] if 'answer' in result else "",
            }]

        }
        return data

    # Extract the JSON block from the response.
    def extract_json(self, text):
        matches = re.findall(r"\{.*\}", text, re.DOTALL)
        if not matches:
            raise ValueError("No JSON found in model output")
        for m in reversed(matches):
            try:
                return json.loads(m)
            except json.JSONDecodeError:
                continue
        raise ValueError("No valid JSON found in model output")


    def get_filter_answer_llm(self, query: str, context: str, model: str,
                              ignore_history: bool = True):
        """
        This function takes a filtering question and asks it of the context.
        The response shows the chain of through and an
        "answer" which can be of the values "yes", "no" or "unsure".

        Note: you should ensure that the query is a "yes/no" question.

        """

        prompt_template = """
        You are an assistant being asked a question based on the following
        segments of a document that have been provided below and described as
        your "context". The context comes from research papers and is split up
        into chunks with a row of "-" as spaces in between chunks. From this,
        you will provide an accurate answer to the question provided.
        ----------------------------------------
        The context is:
        {context}

        ----------------------------------------
        The question is:
        {query}

        ----------------------------------------
        You should provide your answer as a JSON object with the following
        structure:
        {{
            "chain of thought": string,
            "answer": string
        }}

        Where "answer is the answer to the question

        ensure that the JSON response is formatted correctly as stated above and
        has {{}} at the start and end of the response.

        For example:

        question: "Does this application use LSTMs?"

        context: "LSTMs were used to help with the vanishing gradient problem."

        JSON response:
        {{
            "chain of thought": "The research paper mentions LSTMs in the context of Vanishing gradient problem.",
            "answer": "yes, the application uses lstms to solve the vanishing gradient problem."
        }}

        """

        if ignore_history:
            messages = []
        else:
            if self.history == []:
                self.history.append(
                    {"role": "system", "content": "You are a helpful assistant"})
            messages = self.history.copy()

        messages.append({"role": "user", "content": prompt_template.format(query=query, context=context)})

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Tokenize prompt
        inputs = tokenizer(text, return_tensors="pt").to(model_instruct.device)

        # Generate output
        with torch.no_grad():
            output_ids = model_instruct.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode output
        response_text = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        if not ignore_history:
            self.history.append({"role": "user", "content": query})
            self.history.append({"role": "system", "content": response_text})
            history = self.history
        else:
            messages.append({"role": "system", "content": response_text})
            history = messages

        result_json = self.extract_json(response_text)

        return result_json, history



In [17]:
from pypdf import PdfReader

class PDFLoaderWithSplit:
    """Re-implementation of PyPDFLoader.load_and_split(),
       which is unfortunately depracated from LangChain!"""

    def __init__(self, file_path, chunk_size=200, chunk_overlap=20):
        """
        Args:
            file_path (str): Path to PDF
            chunk_size (int): Number of words per chunk
            chunk_overlap (int): Overlap between chunks (in words)
        """
        self.file_path = file_path
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def load(self):
        """Load PDF pages as list of strings"""
        reader = PdfReader(self.file_path)
        pages = []
        for page in reader.pages:
            text = page.extract_text()
            if text and text.strip():
                pages.append(text.strip())
        return pages

    def split_text(self, texts=None):
        """Split text into overlapping chunks"""
        if texts is None:
            texts = self.load()

        chunks = []
        for text in texts:
            words = text.split()
            start = 0
            while start < len(words):
                end = start + self.chunk_size
                chunk = " ".join(words[start:end])
                if chunk.strip():
                    chunks.append(chunk)
                start += self.chunk_size - self.chunk_overlap
        return chunks

    def load_and_split(self):
        """Load PDF and return list of chunks"""
        pages = self.load()
        chunks = self.split_text(pages)
        return chunks

In [14]:

DOCS_PATH = '/mnt/drive/MyDrive/documents'
def chunking_and_storing_in_db():
    chunker_embedder = ChunkerEmbedder()
    for file_name in os.listdir(DOCS_PATH):
        # Check if it's a file (not a folder)
        if os.path.isfile(os.path.join(DOCS_PATH, file_name)):
            loader = PDFLoaderWithSplit(os.path.join(DOCS_PATH, file_name))
            doc = loader.load_and_split()
            print(file_name, type(doc), len(doc))
            chunks = chunker_embedder.chunk_document(doc)



In [19]:
# On the first run, set flag to True to populate the DB with the embedded chunks.
perform_chunking = False

chunker_embedder = ChunkerEmbedder()
query = "Are graphs used?"
if perform_chunking:
    if "documents" in chunker_embedder.db.list_collection_names():
        chunker_embedder.db.documents.drop()

    # Insert a dummy doc to force collection creation
    chunker_embedder.db.documents.insert_one({"_init": True})

    # Recreate the index
    chunker_embedder.db.documents.create_search_index({
        "name": "vec_idx",
        "type": "vectorSearch",
        "definition": {
            "fields": [{
                "type": "vector",
                "path": "embeddings",
                "numDimensions": 1024,
                "similarity": "cosine"
            }]
        }
    })
    import time
    while True:
        indexes = list(chunker_embedder.db.documents.list_search_indexes())
        if indexes and indexes[0]['status'] == 'READY':
            print("Index ready")
            break
        print("Waiting...")
        time.sleep(5)
    chunker_embedder.db.documents.delete_one({"_init": True})
    chunking_and_storing_in_db()

if not perform_chunking:
    res, doc_ids, contents = chunker_embedder.vector_search(query=query)
    context_prompt = chunker_embedder.create_prompt_text(contents)
    model_params = {'model': MODEL_INSTRUCT}
    data = chunker_embedder.ask_question(query, context_prompt, model_params=model_params)
    print(data['result']['chain of thought'])
    print(data['result']['answer'])

The context discusses neural networks operating on graphs, which implies the use of graphs. Additionally, it mentions 'graph structure' and 'graphs' multiple times, indicating that graphs are indeed utilized in the discussed systems and methods.
Yes, graphs are used in the context provided.


TODOs and suggested exercises:
* Once you have performed the necessary steps to setup the database and
pipeline. Are you getting decent performance for searches? Hint: Look at what is being populated in the VectorDB. What issues do you notice with the embeddings? Fix it, wipe the DB and then perform the Vector Search again with the correct embeddings.
* Once correct, try printing out the top scoring retrived chunks and their associated similarity scores. This can be done with the loop `for r in res:
        print(r['score'], r['content'][:80])`, for example.
        Do they seem aligned with the query?
* If you were to use a larger embedding model, such as Qwen3-Embedding-4B, what will need to change in the vector index and the embeddings? Any trade-offs?
* Experiment with different collections of documents of your choice to evaluate the technique.
* What happens if you ask a question which isn't a yes/no question?
* What happens if your question is out of context "Is the sky blue?" ?
* Consider Re-Implementing the pipeline with FAISS (Facebook AI Similarity Search) instead of MongoDB.
